# Gov24 캡차 이미지 수집

`https://plus.gov.kr/api/member/v1.0/nlogin/captcha` API를 호출하여 캡차 이미지를 다운로드합니다.

In [5]:
# 필요한 라이브러리 임포트
import requests
from pathlib import Path
from datetime import datetime
import time
from PIL import Image
from io import BytesIO

In [6]:
# 저장 경로 설정
base_path = Path("captcha_data/gov24/0/images/draft")
base_path.mkdir(parents=True, exist_ok=True)

print(f"저장 경로: {base_path.absolute()}")

저장 경로: c:\work\hyper-captcha-resolver\captcha_data\gov24\0\images\draft


In [7]:
def download_captcha(save_path: Path, index: int, total: int) -> bool:
    """
    캡차 이미지를 다운로드하고 저장합니다.
    
    Args:
        save_path: 저장할 경로
        index: 현재 인덱스
        total: 전체 개수
    
    Returns:
        성공 여부
    """
    url = "https://plus.gov.kr/api/member/v1.0/nlogin/captcha"
    
    try:
        # API 호출
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # 타임스탬프 기반 파일명 생성
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        filename = f"{timestamp}.png"
        filepath = save_path / filename
        
        # 이미지로 변환 후 PNG로 저장
        image = Image.open(BytesIO(response.content))
        image.save(filepath, format="PNG")
        
        print(f"[{index + 1}/{total}] 저장 완료: {filename}")
        return True
        
    except Exception as e:
        print(f"[{index + 1}/{total}] 오류 발생: {e}")
        return False

In [8]:
# 다운로드 설정
TARGET_COUNT = 200  # 다운로드할 이미지 개수
DELAY = 0.5         # 요청 간 대기 시간 (초)

print(f"캡차 이미지 {TARGET_COUNT}개 다운로드 시작...")
print(f"요청 간격: {DELAY}초")
print("-" * 50)

캡차 이미지 200개 다운로드 시작...
요청 간격: 0.5초
--------------------------------------------------


In [ ]:
# 이미지 다운로드 실행
success_count = 0
fail_count = 0

for i in range(TARGET_COUNT):
    if download_captcha(base_path, i, TARGET_COUNT):
        success_count += 1
    else:
        fail_count += 1
    
    # 마지막 요청이 아니면 대기
    if i < TARGET_COUNT - 1:
        time.sleep(DELAY)

print("-" * 50)
print(f"\n다운로드 완료!")
print(f"성공: {success_count}개")
print(f"실패: {fail_count}개")
print(f"저장 위치: {base_path.absolute()}")

## 캡차 이미지 인식 및 파일명 변경

학습된 모델을 사용하여 draft 폴더의 이미지를 인식하고, 예측된 레이블로 파일명을 변경합니다.

In [1]:
import os
from pathlib import Path
import tensorflow as tf
from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.keras_core import KerasModel

captcha_id = 'gov24'
backend = 'keras'
batch_size = 32

draft_image_dir = "captcha_data/gov24/0/images/draft"
draft_image_files = sorted([str(p) for p in Path(draft_image_dir).glob("*.png") if len(p.name) > 10])
print(f"Draft 이미지 개수: {len(draft_image_files)}")

model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
keras_model: KerasModel = model
matched = 0
pred_img_path_list = keras_model.train_data.get_data_files(train=False)
# pred_labels = model.train_data.get_labels(train=False)
pred_dataset = tf.data.Dataset.from_tensor_slices((pred_img_path_list))
pred_dataset = (
    pred_dataset
    .map(keras_model.encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Load prediction model if not loaded
keras_model.load_prediction_model()

for idx, img_path in enumerate(draft_image_files):
    pred, confidence = engine.predict(model=model, image_path=img_path, verbose=0)
    # rename draft image with predicted text
    new_image_path = os.path.join(draft_image_dir, pred + ".png")
    if(os.path.exists(new_image_path)):
        continue
    Path(img_path).rename(new_image_path)
    print(f"[{idx + 1}/{len(draft_image_files)}] image_path : {img_path}")
    print(f"pred : {pred}")
    print(f"confidence : {confidence:.4f}")
    print("new_image_path : ", new_image_path)
    print("Done!")


Draft 이미지 개수: 106


